"""
For our downloaded molecular dynamics MD22/MD17 datasets, what frame stride gives bins that are different enough 
to be interesting, but not so different that the dynamics become super jumpy and weird??

- TLDR; Which stride is best.

Right we are using Jaccard to determine this, where a score of 1 means edges in bins are identical.

"""

In [10]:
from dataclasses import replace
from pathlib import Path
import numpy as np
import torch

import os, sys
sys.path.append("/home/akapociu/ift/interactiondynamics")
print(os.getcwd())
print(sys.path[-1])

/home/akapociu/ift/interactiondynamics/analysis
/home/akapociu/ift/interactiondynamics


In [ ]:
from data.md22_binned import MD22BinnedConfig, MD22BinnedDataset

out_path = Path("/home/akapociu/ift/interactiondynamics/results/md22_stride_calibration.jsonl")
if out_path.exists():
    out_path.unlink()

In [12]:
# turns one batch into a set of edges (get {(0,1), (2,3), (1,0)} ) 
def edge_set(batch):
    return set(zip(batch.src.cpu().tolist(), batch.dst.cpu().tolist()))

# quick helper to complement jaccard score (just in case!!)
def avg_edges_per_bin(ds, split="train"):
    bins = list(ds.bins(split))
    if len(bins) == 0:
        return None
    return float(np.mean([batch.src.numel() for batch in bins]))

# compute avg jaccard similarity across time 
def mean_jaccard_for_dataset(ds, split="train"):
    bins = list(ds.bins(split))
    edge_sets = [edge_set(batch) for batch in bins] # use func above!! ↑↑ e.g. edge_set[0]={(1,1), (2,2)}

    if len(edge_sets) < 2: # for first bin
        return None

    vals = []
    for prev, curr in zip(edge_sets[:-1], edge_sets[1:]): #for neighboring pairs of bins
        inter = len(prev & curr) # edges both bin share
        union = len(prev | curr) # edges in either bin
        vals.append(inter / union if union > 0 else 1.0) # 3 edges divided by 4 edges! .75 score wow

    return float(np.mean(vals)) # take average of whole dataset, so all bins!!


# MAIN FUNCTION
# range is .55 - .8, this is just a guess of some solid variety w/o getting too crazy
def calibrate_md22_stride(
    npz_path,
    *,
    device,
    base_cfg=None,
    candidate_strides=(32, 64, 128, 256, 512, 1024, 2056), # I know IFT hates tiny strides, so let's see what happens here
    target_low=0.55, 
    target_high=0.80, 
    min_total_bins=256,
    max_total_bins=1024, # this probably isn't helping too much oh well... 512?
    split="train",
):
    """
    1. Try all the cadidate_stride options
    2. Build dataset for each one
    3. Measure avg edge similarity w/ jaccard
    4. Keep diagnostics
    5. Choose best stride

    Returns:
        best_cfg, diagnostics_rows
    """
    stem = Path(npz_path).stem # molecule name (file name w/ .npz)


    #default configggg -> from main MD22_binned.py script
    if base_cfg is None: 
        base_cfg = MD22BinnedConfig(
            name=stem,
            npz_path=str(npz_path),
            event_mode="distance",          # or whatever you actually want
            distance_threshold=5.0,
            distance_change_threshold=0.1,
            distance_change_use_absolute=True,
            knn_k=4,
            observation_noise_pos=0.0,
            observation_noise_force=0.0,
            min_edges_per_bin=1,
            device=device,
        )

    rows = []

    # try each stride
    for stride in candidate_strides:
        cfg = replace( #override what you want
            base_cfg,
            name=stem,
            npz_path=str(npz_path),
            frame_stride=int(stride),
            max_frames=max_total_bins,   # optional cap so huge molecules don't explode
        )

        try:
            ds = MD22BinnedDataset(cfg) # construct dataset
            spec = ds.spec() # get metadata
            mj = mean_jaccard_for_dataset(ds, split=split) # compute jaccard
            avg_edges = avg_edges_per_bin(ds, split=split)

            # record result for that stride candidate
            row = {
                "molecule": stem,
                "stride": int(stride),
                "event_mode": cfg.event_mode,
                "num_bins": int(spec.num_bins),
                "num_nodes": int(spec.num_nodes),
                "mean_jaccard": mj,
                "avg_edges_per_bin": avg_edges,
                "valid_bins": (spec.num_bins is not None and spec.num_bins >= min_total_bins), # safety just in case
            }
            rows.append(row)

        # hi error handling
        except Exception as e:
            rows.append({
                "molecule": stem,
                "stride": int(stride),
                "event_mode": base_cfg.event_mode,
                "num_bins": None,
                "num_nodes": None,
                "mean_jaccard": None,
                "valid_bins": False,
                "avg_edges_per_bin": None,
                "error": f"{type(e).__name__}: {e}",
            })

    # valid candidates
    valid = [
        r for r in rows
        if r["mean_jaccard"] is not None and r["valid_bins"]
    ]

    if not valid:
        raise RuntimeError(f"No valid stride candidates for {stem}")

    # prefer candidates inside the target band
    in_band = [
        r for r in valid
        if target_low <= r["mean_jaccard"] <= target_high
    ]

    target_mid = 0.5 * (target_low + target_high)

    def score(r):
        # primary: closeness to target midpoint
        # secondary: prefer smaller stride if tied
        # tertiary: prefer more bins
        return (
            abs(r["mean_jaccard"] - target_mid),
            r["stride"],
            -r["num_bins"],
        )

    # winner
    if in_band:
        chosen = sorted(in_band, key=score)[0]
    else:
        # fallback: closest Jaccard, still respecting min bins
        chosen = sorted(valid, key=score)[0]

    #new cfg build from results
    best_cfg = replace(
        base_cfg,
        name=stem,
        npz_path=str(npz_path),
        frame_stride=int(chosen["stride"]),
        max_frames=max_total_bins,
    )

    return best_cfg, rows

In [13]:
# one dictionary per line to file shoutout JSONL
def append_jsonl(path, row):
    import json
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row) + "\n")

md22_npz_paths = [
    "/home/akapociu/ift/interactiondynamics/data/MD_DATA/uracil.npz",
    "/home/akapociu/ift/interactiondynamics/data/MD_DATA/naphthalene.npz",
    "/home/akapociu/ift/interactiondynamics/data/MD_DATA/stachyose.npz",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# collects best configs
chosen_cfgs = []

# loop over molecules
for npz_path in md22_npz_paths:
    best_cfg, rows = calibrate_md22_stride( #runs whole stride search for one molecule
        npz_path,
        device=device,
        base_cfg=MD22BinnedConfig(
            name=Path(npz_path).stem,
            npz_path=npz_path,
            event_mode="distance",
            distance_threshold=5.0,
            observation_noise_pos=0.0,
            observation_noise_force=0.0,
            min_edges_per_bin=1,
            device=device,
        ),
        candidate_strides=(32, 64, 128, 256, 512, 1024, 2056),
        target_low=0.55,
        target_high=0.80,
        min_total_bins=256,
        max_total_bins=512,
    )

    chosen_cfgs.append(best_cfg)

    # SAVE IT
    for row in rows:
        append_jsonl(out_path, row)

    print(
        f"{Path(npz_path).stem}: chose stride={best_cfg.frame_stride}, "
        f"event_mode={best_cfg.event_mode}"
    )

uracil: chose stride=64, event_mode=distance
naphthalene: chose stride=32, event_mode=distance
stachyose: chose stride=64, event_mode=distance
